# Classificação de textos: TF-IDF e embeddings

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_08/04_classificacao_textual_tfidf_e_embeddings.ipynb)

TF-IDF representa documentos por palavras ou sequências curtas, ponderadas pela
frequência no documento e raridade na coleção. Embeddings representam proximidades
aprendidas em espaços densos. Ambos são representações parciais, não o texto em si.

![TF-IDF privilegia termos discriminativos; embeddings aproximam usos semanticamente semelhantes.](imagens/04_representacoes.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_08'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('sklearn', 'scikit-learn>=1.5,<2')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Um modelo textual reproduzível e offline

Ajustamos o vocabulário somente no treino por meio de um `Pipeline`; assim o teste
não informa quais termos devem existir. Usamos regressão logística por oferecer
coeficientes inspecionáveis. O objetivo não é decorar parâmetros, mas acompanhar a
cadeia texto → vetor → modelo → erro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay

dados = pd.read_csv("dados/documentos_classificacao.csv")
treino, teste = train_test_split(dados, test_size=.25, random_state=42, stratify=dados["classe_alvo"])
classificador_textual = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
    ("modelo", LogisticRegression(max_iter=1000, random_state=42)),
])
classificador_textual.fit(treino["texto"], treino["classe_alvo"])
previsto_texto = classificador_textual.predict(teste["texto"])
metricas_texto = pd.Series({
    "acurácia": accuracy_score(teste["classe_alvo"], previsto_texto),
    "precisão": precision_score(teste["classe_alvo"], previsto_texto, pos_label="relevante"),
    "revocação": recall_score(teste["classe_alvo"], previsto_texto, pos_label="relevante"),
    "F1": f1_score(teste["classe_alvo"], previsto_texto, pos_label="relevante"),
})
display(metricas_texto.to_frame("TF-IDF + logística").T.round(3))

## 2. O que o modelo textual utilizou?

Coeficientes positivos aproximam termos associados a “relevante” no treino;
negativos, a “não relevante”. Eles não medem importância histórica nem significado
intrínseco. Termos de período podem denunciar atalho: desempenho alto apoiado em
marcas contextuais que talvez mudem fora desta coleção.

In [ ]:
termos = classificador_textual.named_steps["tfidf"].get_feature_names_out()
coef = classificador_textual.named_steps["modelo"].coef_[0]
extremos = pd.DataFrame({"termo": termos, "coeficiente": coef})
display(pd.concat([extremos.nsmallest(8, "coeficiente"), extremos.nlargest(8, "coeficiente")]).sort_values("coeficiente"))

avaliacao_texto = teste.assign(previsto=previsto_texto)
erros_texto = avaliacao_texto[avaliacao_texto["classe_alvo"] != avaliacao_texto["previsto"]]
display(erros_texto[["id_documento", "texto", "classe_alvo", "previsto"]].head(10))

## 3. Onde entram embeddings?

Um embedding troca milhares de contagens esparsas por vetores densos aprendidos em
grandes coleções. Pode aproximar paráfrases sem palavras idênticas, mas carrega
vieses, domínio, versão e custo do modelo de origem. Nesta unidade ele é introduzido
conceitualmente; executar um modelo remoto tornaria a aula dependente de rede e
enviaria textos a terceiros. Uma comparação controlada pertence à Unidade 12.

### U08-A05 — Atividade — termos, atalhos e erros

Inspecione os termos extremos e todos os erros textuais. Identifique um termo
substantivo, um possível atalho e um caso ambíguo. Proponha uma nova rodada de
anotação ou teste temporal capaz de desafiar o resultado.

**Dinâmica (30 minutos):** duplas; uma pessoa lê a tabela e outra retorna ao texto.
Troquem papéis antes de formular a proposta.

**Resposta:** escreva aqui.

## Passagem

A oficina reúne definição, avaliação e leitura dos erros. Classificar só entra no
projeto quando a categoria e o custo de seus erros forem defensáveis.